# Jinavera FDE Encoder

- Pre-requisites
  - Download the codes of embedding model of `jina_embeddings_v4` to import `modeling_jina_embeddings_v4.JinaEmbeddingsV4Model`
    - only python files and config files, and exclude model files
  - `muvfde` installation
  - `fde.py`
    - Handle muvfde configs according to the prompt type between `query` and `passage` input

- Future works
  - Can publish the FDE Encoder models corresponding to the matched embedding models like `jinavera`
  - `blipvera, florencevera, sigvera, ...` 

- Known issue
    - Just use `flash_attention_v2` for `_attn_implementation` of `config.json`
    - ~~qwen2_5_vl.py: Updated the attention implementation to use "sdpa" instead of relying on the config's `_attn_implementation` attribute.~~
    - ~~jina_embeddings_v4/config.json: Update the attention implementation(`_attn_implementation`) as `flash_attention_v2` to `sdpa`.~~

In [1]:
import torch
import peft
# import flash_attn

/home/ed/git/kwazii/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
(
    torch.__version__,
    peft.__version__,
    # flash_attn.__version__
)

('2.6.0+cu118', '0.18.0')

In [3]:
import transformers

In [4]:
import torch
from transformers import AutoModel, AutoTokenizer
from jina_embeddings_v4.modeling_jina_embeddings_v4 import JinaEmbeddingsV4Model


In [5]:
_ENCODER_MODEL_ID = 'jina-embeddings-v4'

In [6]:
# from tunip.path_utils import services as path_services
# from tunip.service_config import get_service_config

# service_config = get_service_config()
# path_service = path_services(service_config)
# encoder_model_path = path_service.build(f'/user/{service_config.username}/mart/llms/{_ENCODER_MODEL_ID}')

# encoder_model_path

In [7]:
encoder_model_path = '/mnt/d/temp/user/ed/mart/llms/jina-embeddings-v4'

In [8]:
torch.cuda.empty_cache()

In [9]:
# from jina_embeddings_v4.modeling_jina_embeddings_v4 import JinaEmbeddingsV4Model

# model = JinaEmbeddingsV4Model.from_pretrained(encoder_model_path, trust_remote_code=False)

# # Error
# # amodel = AutoModel.from_pretrained(encoder_model_path, trust_remote_code=False)

## Jinavera

In [10]:
torch.cuda.is_available(), torch.cuda.get_device_name(0)

(True, 'NVIDIA GeForce RTX 3090 Ti')

In [11]:
import numpy as np
import muvfde
import fde  # local import

In [12]:
# muvfde configuration
num_repetitions = 20
num_simhash_projections = 5
projection_dimension = 16
final_projection_dimension = None
seed = 1221

In [13]:
fde_model = None

In [14]:
class Jinavera(torch.nn.Module):

    def __init__(self, encoder_model_path: str):
        super().__init__()
        # self.encoder_model = AutoModel.from_pretrained(encoder_model_path)
        self.encoder_model = JinaEmbeddingsV4Model.from_pretrained(encoder_model_path, trust_remote_code=False).to('cuda')

        # TODO(cloudjo21) keep the unit test tokenizer loading with local_files_only=True
        # self.tokenizer = AutoTokenizer.from_pretrained(encoder_model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(encoder_model_path, trust_remote_code=False, use_fast=True, local_files_only=True)
        self.fde_configs = {
            fde.PromptType.QUERY: fde.FdeConfig.apply_with_prompt_type(fde.PromptType.QUERY),
            fde.PromptType.PASSAGE: fde.FdeConfig.apply_with_prompt_type(fde.PromptType.PASSAGE)
        }

    def encode(self, texts: list[str], prompt_type: fde.PromptType = fde.PromptType.QUERY) -> np.ndarray:
        # TODO(cloudjo21) keep the unit test tokenizer loading with local_files_only=True
        tokens = self.tokenizer(texts, return_tensors="pt", padding=True).to('cuda')
        # tokens = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True)

        token_embs = self.encoder_model.encode_text(
            texts=texts,
            task="retrieval",
            return_multivector=True,
            prompt_name=prompt_type.value
        )
        # TODO(ed-on-firecuda): Use the encoder_model's encode_image method if available
        # last_hidden_states = self.encoder_model.get_last_hidden_states("retrieval", tokens.input_ids, tokens.attention_mask)
        # token_embs = self.encoder_model.get_multi_vector_embeddings(hidden_states=last_hidden_states, attention_mask=tokens.attention_mask, task_label="retrieval")
        # outputs = self.encoder_model.get_text_features(**inputs)

        fde_out = []
        for mat in token_embs:
            print(f"#### mat shape: {mat.shape}")
            fde_out.append(
                muvfde.generate_fixed_dimensional_encoding(
                    # mat.cpu().to(torch.float32),
                    # mat.cpu().numpy().to(torch.float32),
                    mat.to(dtype=torch.float32, device='cpu').numpy(),
                    self.fde_configs[prompt_type]
                )
            )
            print(f"#### fde_out shape: {fde_out[-1].shape}")
        return np.stack(fde_out, axis=0)

    def similarity(self, a: np.ndarray, b: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(a @ b.T)

    def __finalize__(self):
        del self.encoder_model
        torch.cuda.empty_cache()

In [15]:
if fde_model:
    del fde_model
    torch.cuda.empty_cache()
fde_model = Jinavera(encoder_model_path)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.65it/s]


In [16]:
fde_vector = fde_model.encode(["Hello, world!", "Good-bye, world!"])
fde_vector.shape

Encoding texts...:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  1.61it/s]

#### mat shape: torch.Size([6, 128])
#### fde_out shape: (10240,)
#### mat shape: torch.Size([8, 128])
#### fde_out shape: (10240,)


(2, 10240)

In [17]:
import peft

In [18]:
# Expected FDE dimension calculation
fde_dimension = num_repetitions * (2**num_simhash_projections) * projection_dimension
fde_dimension

10240

In [19]:
np.all(fde_vector[0] == 0.), fde_vector[0].any()

(np.False_, np.True_)

In [20]:
np.nonzero(fde_vector[0])[0].size, np.nonzero(fde_vector[1])[0].size

(1616, 2080)

In [21]:
text_embeddings = fde_model.encoder_model.encode_text(
    texts=["Hello, world!", "Good-bye, world!"],
    task="retrieval",
    return_multivector=True,
    prompt_name="query")


Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  4.06it/s]


In [22]:
text_embeddings[0].shape

torch.Size([6, 128])

In [23]:
from transformers.modeling_flash_attention_utils import _flash_attention_forward

In [24]:
# ========================
# 1. Retrieval Task
# ========================
# Configure truncate_dim, max_length (for texts), max_pixels (for images), vector_type, batch_size in the encode function if needed

# Encode query
with torch.no_grad():
    model.config.use_flash_attention_2 = False
    # Encode text for retrieval
    query_embeddings = model.encode_text(
        texts=["Overview of climate change impacts on coastal cities"],
        task="retrieval",
        prompt_name="query",
    )

NameError: name 'model' is not defined

In [ ]:
# Encode passage (text)
passage_embeddings = model.encode_text(
    texts=[
        "Climate change has led to rising sea levels, increased frequency of extreme weather events..."
    ],
    task="retrieval",
    prompt_name="passage",
)

Encoding texts...: 100%|██████████| 1/1 [00:02<00:00,  2.62s/it]


# Model Study

In [ ]:
encoder_model_id = 'jina-embeddings-v4'

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

from jina_embeddings_v4.modeling_jina_embeddings_v4 import JinaEmbeddingsV4Model

from tunip.path_utils import services as path_services
from tunip.service_config import get_service_config

service_config = get_service_config()
path_service = path_services(service_config)
encoder_model_path = path_service.build(f'/user/{service_config.username}/mart/llms/{encoder_model_id}')


/home/ed/env/uvpy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(encoder_model_path, trust_remote_code=False, use_fast=True, local_files_only=True)
model = JinaEmbeddingsV4Model.from_pretrained(encoder_model_path, trust_remote_code=False)

# Error
# amodel = AutoModel.from_pretrained(encoder_model_path, trust_remote_code=False)

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.20it/s]


In [ ]:
type(model)

peft.peft_model.PeftModelForFeatureExtraction

In [ ]:
model.__dict__

{'training': False,
 '_parameters': {},
 '_buffers': {},
 '_non_persistent_buffers_set': set(),
 '_backward_pre_hooks': OrderedDict(),
 '_backward_hooks': OrderedDict(),
 '_is_full_backward_hook': None,
 '_forward_hooks': OrderedDict(),
 '_forward_hooks_with_kwargs': OrderedDict(),
 '_forward_hooks_always_called': OrderedDict(),
 '_forward_pre_hooks': OrderedDict(),
 '_forward_pre_hooks_with_kwargs': OrderedDict(),
 '_state_dict_hooks': OrderedDict(),
 '_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_post_hooks': OrderedDict(),
 '_modules': {'base_model': LoraModel(
    (model): JinaEmbeddingsV4Model(
      (model): Qwen2_5_VLModel(
        (visual): Qwen2_5_VisionTransformerPretrainedModel(
          (patch_embed): Qwen2_5_VisionPatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
          (blocks): ModuleL

In [ ]:
# model.base_model.model.model
model.base_model.model.model.language_model.config

Qwen2_5_VLTextConfig {
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "image_token_id": null,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"

In [ ]:
import ast

with open("/mnt/d/temp/user/ed/mart/llms/jina-embeddings-v4/modeling_jina_embeddings_v4.py", "r") as f:
    tree = ast.parse(f.read())

classes = [node.name for node in ast.walk(tree) if isinstance(node, ast.ClassDef)]
functions = [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]

print("Classes:", classes)
print("Functions:", functions)


Classes: ['PromptType', 'JinaEmbeddingsV4Processor', 'JinaEmbeddingsV4ModelOutput', 'JinaEmbeddingsV4Model']
Functions: ['__init__', 'process_images', 'process_texts', '__init__', 'task', 'task', 'get_last_hidden_states', '_init_projection_layer', 'get_single_vector_embeddings', 'get_multi_vector_embeddings', '_input_has_image', 'forward', '_process_batches', '_validate_encoding_params', '_validate_task', 'encode_text', '_load_images_if_needed', 'encode_image', 'from_pretrained', 'task_getter', 'task_setter']


In [ ]:
# model = AutoModel.from_pretrained(encoder_model_path, torch_dtype=torch.float16, trust_remote_code=False)
# # model = AutoModel.from_pretrained(encoder_model_path)

In [ ]:
tokens = tokenizer(["Hello, world!"], return_tensors="pt")
tokens = tokenizer(["Good-bye, world!"], return_tensors="pt")
tokens

{'input_ids': tensor([[15216,  1455,  9011,    11,  1879,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [ ]:
# infer single- and multi-vector embeddings at the same time
embedding_output = model(**tokens, task_label="retrieval")
embedding_output

JinaEmbeddingsV4ModelOutput(vlm_last_hidden_states=None, single_vec_emb=tensor([[ 0.0160, -0.0586,  0.0342,  ...,  0.0076,  0.0025,  0.0010]],
       dtype=torch.bfloat16), multi_vec_emb=tensor([[[-6.1279e-02,  1.2012e-01,  1.1230e-01, -2.5330e-03,  7.9102e-02,
          -2.3535e-01, -3.8086e-02,  5.6885e-02, -1.3916e-02,  1.1670e-01,
           5.0049e-03,  4.0039e-02,  4.0283e-02,  9.1797e-02, -1.1377e-01,
          -1.0742e-01,  1.2500e-01, -6.6406e-02, -1.0059e-01, -2.4707e-01,
           1.0925e-02,  1.5820e-01,  7.9590e-02, -2.0996e-01,  7.0801e-02,
           3.6865e-02,  2.1729e-02,  4.0527e-02, -3.6774e-03, -2.6855e-02,
          -4.9438e-03, -2.9663e-02, -4.4556e-03,  1.2012e-01,  5.4321e-03,
           7.2861e-04, -1.2012e-01, -6.1768e-02, -4.3457e-02, -6.6895e-02,
          -4.1504e-02,  8.0078e-02, -8.1543e-02, -3.3936e-02,  9.1309e-02,
          -8.1543e-02, -9.3750e-02, -1.7700e-02,  1.8921e-02, -1.1670e-01,
          -9.5703e-02,  1.0547e-01, -1.5015e-02,  6.8970e-03, -

In [ ]:
embedding_output.multi_vec_emb.shape, tokens

(torch.Size([1, 6, 128]),
 {'input_ids': tensor([[15216,  1455,  9011,    11,  1879,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])})

In [ ]:
tokens.attention_mask.shape, tokens.attention_mask.dtype

(torch.Size([1, 6]), torch.int64)

In [ ]:
last_hidden_states = model.get_last_hidden_states("retrieval", tokens.input_ids, tokens.attention_mask)

In [ ]:
last_hidden_states.shape, last_hidden_states.dtype

(torch.Size([1, 6, 2048]), torch.bfloat16)

In [ ]:
m_vectors = model.get_multi_vector_embeddings(hidden_states=last_hidden_states, attention_mask=tokens.attention_mask, task_label="retrieval")

In [ ]:
m_vectors.shape

torch.Size([1, 6, 128])

In [ ]:
m_vectors

tensor([[[-6.1279e-02,  1.2012e-01,  1.1230e-01, -2.5330e-03,  7.9102e-02,
          -2.3535e-01, -3.8086e-02,  5.6885e-02, -1.3916e-02,  1.1670e-01,
           5.0049e-03,  4.0039e-02,  4.0283e-02,  9.1797e-02, -1.1377e-01,
          -1.0742e-01,  1.2500e-01, -6.6406e-02, -1.0059e-01, -2.4707e-01,
           1.0925e-02,  1.5820e-01,  7.9590e-02, -2.0996e-01,  7.0801e-02,
           3.6865e-02,  2.1729e-02,  4.0527e-02, -3.6774e-03, -2.6855e-02,
          -4.9438e-03, -2.9663e-02, -4.4556e-03,  1.2012e-01,  5.4321e-03,
           7.2861e-04, -1.2012e-01, -6.1768e-02, -4.3457e-02, -6.6895e-02,
          -4.1504e-02,  8.0078e-02, -8.1543e-02, -3.3936e-02,  9.1309e-02,
          -8.1543e-02, -9.3750e-02, -1.7700e-02,  1.8921e-02, -1.1670e-01,
          -9.5703e-02,  1.0547e-01, -1.5015e-02,  6.8970e-03, -2.0801e-01,
          -1.1230e-01, -2.5146e-02,  4.5410e-02,  1.2793e-01,  1.0742e-01,
           1.6797e-01,  1.4160e-01, -1.9409e-02, -1.1621e-01, -8.0566e-02,
          -2.1118e-02, -1

In [ ]:
import muvfde

a = muvfde.fixed_dimensional_encoding_config()
type(a)

muvfde.muvfde_ext.fixed_dimensional_encoding_config

In [ ]:
import torch
import torch.nn.functional as F

# 간단한 예제 설정
batch_size = 2
seq_length = 4
hidden_size = 6
multi_vector_dim = 3

# 입력 데이터 가정
hidden_states = torch.randn(batch_size, seq_length, hidden_size)
attention_mask = torch.tensor([
    [1, 1, 1, 0],  # 첫 번째 시퀀스: 3개 토큰이 유효
    [1, 1, 0, 0]   # 두 번째 시퀀스: 2개 토큰이 유효
], dtype=torch.float)

# Multi-vector projector (Linear layer) 시뮬레이션
projector = torch.nn.Linear(hidden_size, multi_vector_dim)

print("=== 입력 데이터 ===")
print(f"hidden_states shape: {hidden_states.shape}")
print(f"hidden_states:\n{hidden_states}")
print(f"\nattention_mask shape: {attention_mask.shape}")
print(f"attention_mask:\n{attention_mask}")

print("\n=== 처리 과정 ===")

# 1. Multi-vector projection
multi_vec_emb = projector(hidden_states)
print(f"1. After projection - shape: {multi_vec_emb.shape}")
print(f"multi_vec_emb:\n{multi_vec_emb}")

# 2. L2 정규화
multi_vec_emb_normalized = F.normalize(multi_vec_emb, dim=-1)
print(f"\n2. After normalization - shape: {multi_vec_emb_normalized.shape}")
print(f"multi_vec_emb_normalized:\n{multi_vec_emb_normalized}")
print(f"L2 norm check (should be ~1.0):\n{torch.norm(multi_vec_emb_normalized, dim=-1)}")

# 3. Attention mask 적용
attention_mask_expanded = attention_mask.unsqueeze(-1)
print(f"\n3. attention_mask_expanded shape: {attention_mask_expanded.shape}")
print(f"attention_mask_expanded:\n{attention_mask_expanded}")

final_multi_vec_emb = multi_vec_emb_normalized * attention_mask_expanded
print(f"\n4. Final result - shape: {final_multi_vec_emb.shape}")
print(f"final_multi_vec_emb:\n{final_multi_vec_emb}")

print("\n=== 결과 분석 ===")
print("주목할 점:")
print("- Attention mask가 0인 위치의 embedding은 모두 0이 됨")
print("- 첫 번째 시퀀스: 처음 3개 토큰만 유효한 embedding")
print("- 두 번째 시퀀스: 처음 2개 토큰만 유효한 embedding")
print(f"\n첫 번째 시퀀스의 4번째 토큰 (mask=0): {final_multi_vec_emb[0, 3]}")
print(f"두 번째 시퀀스의 3,4번째 토큰 (mask=0): {final_multi_vec_emb[1, 2:]}")

=== 입력 데이터 ===
hidden_states shape: torch.Size([2, 4, 6])
hidden_states:
tensor([[[-0.6895, -0.3327,  0.1462,  0.4762, -1.3516, -0.0908],
         [-1.2228,  1.9312, -0.7281,  0.5368, -1.2094, -0.7224],
         [-1.0879, -0.5468, -0.6006, -1.5985,  1.4535,  0.0784],
         [-1.6526, -0.4505,  1.3269, -1.8333, -0.5545, -0.4907]],

        [[-0.9996, -1.3630, -1.0286,  1.0046,  1.4370,  0.0080],
         [ 1.3868,  0.6043,  1.0980, -2.0853,  0.4006, -1.4924],
         [-0.3755, -0.7222,  0.5961, -0.2314, -0.9168,  0.1007],
         [ 1.8456, -0.3921,  0.1922,  0.8926, -0.3360,  0.6834]]])

attention_mask shape: torch.Size([2, 4])
attention_mask:
tensor([[1., 1., 1., 0.],
        [1., 1., 0., 0.]])

=== 처리 과정 ===
1. After projection - shape: torch.Size([2, 4, 3])
multi_vec_emb:
tensor([[[ 0.4539, -0.9550,  0.4753],
         [-0.5058, -0.8427,  0.1330],
         [-1.6730,  1.0714,  0.7794],
         [-0.4743, -0.2971,  1.0253]],

        [[-0.5562,  0.3647,  1.0950],
         [ 0.0440, 